# Higgs Boson Discovery — End-to-End EDA
**Dataset:** HIGGS UCI (Baldi et al., 2014) — 11M simulated LHC collision events  
**Goal:** Distinguish Higgs boson signal events from background noise using 28 physics features  
**Kernel:** Python 3.11 | uv virtual environment

---

# ── 0. Setup & Constants ──────────────────────────────────────────────────────
import pathlib
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      StratifiedKFold, learning_curve)
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_curve, auc, precision_recall_curve,
                              average_precision_score, f1_score)
warnings.filterwarnings('ignore')

# ── Plot theme
sns.set_theme(style="ticks", palette="husl",
              rc={'figure.figsize': (12, 7), 'font.size': 12,
                  'axes.titlesize': 13, 'axes.labelsize': 12})

# ── Constants
DATA_PATH    = 'data/raw/higgs.csv'
FIGURES_DIR  = pathlib.Path('results/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_SIZE  = 150_000
RANDOM_STATE = 42

# ── Feature names from Baldi et al. (2014), Table 1
COLUMN_NAMES = [
    'label',
    'lepton_pT', 'lepton_eta', 'lepton_phi',
    'missing_energy_magnitude', 'missing_energy_phi',
    'jet1_pT', 'jet1_eta', 'jet1_phi', 'jet1_b_tag',
    'jet2_pT', 'jet2_eta', 'jet2_phi', 'jet2_b_tag',
    'jet3_pT', 'jet3_eta', 'jet3_phi', 'jet3_b_tag',
    'jet4_pT', 'jet4_eta', 'jet4_phi', 'jet4_b_tag',
    'm_jj', 'm_jjj', 'm_lv', 'm_jlv', 'm_bb', 'm_wbb', 'm_wwbb'
]
LOW_LEVEL  = [c for c in COLUMN_NAMES[1:] if not c.startswith('m_')]
HIGH_LEVEL = [c for c in COLUMN_NAMES[1:] if c.startswith('m_')]

SIG_COLOR = '#2196F3'
BKG_COLOR = '#EF5350'

print(f"Environment ready.  Sample size: {SAMPLE_SIZE:,} | Random state: {RANDOM_STATE}")
print(f"Figures directory  : {FIGURES_DIR.resolve()}")
print(f"Low-level features : {len(LOW_LEVEL)}")
print(f"High-level features: {len(HIGH_LEVEL)}")